In [1]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [ ]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "stackoverflow" dataset
dataset_ref = client.dataset("stackoverflow", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "posts_questions" table
table_ref = dataset_ref.table("posts_questions")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe()

,visitorId,visitNumber,visitId,visitStartTime,date,totals,trafficSource,device,geoNetwork,customDimensions,hits,fullVisitorId,userId,clientId,channelGrouping,socialEngagementType
0,<NA>,1,1501591568,1501591568,20170801,"{'visits': 1, 'hits': 1, 'pageviews': 1, 'time...","{'referralPath': None, 'campaign': '(not set)'...","{'browser': 'Chrome', 'browserVersion': 'not a...","{'continent': 'Europe', 'subContinent': 'South...",[],"[{'hitNumber': 1, 'time': 0, 'hour': 5, 'minut...",3418334011779872055,NaN,NaN,Organic Search,Not Socially Engaged
1,<NA>,2,1501589647,1501589647,20170801,"{'visits': 1, 'hits': 1, 'pageviews': 1, 'time...","{'referralPath': '/analytics/web/', 'campaign'...","{'browser': 'Chrome', 'browserVersion': 'not a...","{'continent': 'Asia', 'subContinent': 'Souther...","[{'index': 4, 'value': 'APAC'}]","[{'hitNumber': 1, 'time': 0, 'hour': 5, 'minut...",2474397855041322408,NaN,NaN,Referral,Not Socially Engaged
2,<NA>,1,1501616621,1501616621,20170801,"{'visits': 1, 'hits': 1, 'pageviews': 1, 'time...","{'referralPath': '/analytics/web/', 'campaign'...","{'browser': 'Chrome', 'browserVersion': 'not a...","{'continent': 'Europe', 'subContinent': 'North...","[{'index': 4, 'value': 'EMEA'}]","[{'hitNumber': 1, 'time': 0, 'hour': 12, 'minu...",5870462820713110108,NaN,NaN,Referral,Not Socially Engaged
3,<NA>,1,1501601200,1501601200,20170801,"{'visits': 1, 'hits': 1, 'pageviews': 1, 'time...","{'referralPath': '/analytics/web/', 'campaign'...","{'browser': 'Firefox', 'browserVersion': 'not ...","{'continent': 'Americas', 'subContinent': 'Nor...","[{'index': 4, 'value': 'North America'}]","[{'hitNumber': 1, 'time': 0, 'hour': 8, 'minut...",9397809171349480379,NaN,NaN,Referral,Not Socially Engaged
4,<NA>,1,1501615525,1501615525,20170801,"{'visits': 1, 'hits': 1, 'pageviews': 1, 'time...","{'referralPath': '/analytics/web/', 'campaign'...","{'browser': 'Chrome', 'browserVersion': 'not a...","{'continent': 'Americas', 'subContinent': 'Nor...","[{'index': 4, 'value': 'North America'}]","[{'hitNumber': 1, 'time': 0, 'hour': 12, 'minu...",6089902943184578335,NaN,NaN,Referral,Not Socially Engaged


In [4]:
# SQL Query to calculate how fast questions get answered.
# We define first_query as a multi-line string containing our SQL commands.
first_query = """
              # SELECT the question ID
              SELECT q.id AS q_id,
                  # Calculate the time difference (in seconds) between when the question was asked
                  # and when its answers were created. We use MIN() to get the time of the VERY FIRST answer.
                  MIN(TIMESTAMP_DIFF(a.creation_date, q.creation_date, SECOND)) as time_to_answer
              # From the 'posts_questions' table (aliased as 'q')
              FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
                  # INNER JOIN matches questions with their answers in the 'posts_answers' table (aliased as 'a')
                  INNER JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
              # The connection point: the answer's parent_id must match the question's id
              ON q.id = a.parent_id
              # Only look at questions asked between January 1, 2018 and January 31, 2018
              WHERE q.creation_date >= '2018-01-01' and q.creation_date < '2018-02-01'
              # Group the results by question ID so we find the MIN time per unique question
              GROUP BY q_id
              # Sort the results by the fastest answer time
              ORDER BY time_to_answer
              """

# Run the query using our client, and convert the results into a Pandas DataFrame for easy analysis.
# create_bqstorage_client=False bypasses the BigQuery Storage API which requires special permissions.
first_result = client.query(first_query).result().to_dataframe(create_bqstorage_client=False)

# Calculate the percentage of questions that actually received an answer.
# 1. sum(first_result["time_to_answer"].notnull()) counts how many questions have a valid answer time.
# 2. len(first_result) counts the total number of questions returned by the query.
# 3. Divide them and multiply by 100 to get a percentage.
print("Percentage of answered questions: %s%%" % \
      (sum(first_result["time_to_answer"].notnull()) / len(first_result) * 100))

# Print the total number of questions retrieved.
print("Number of questions:", len(first_result))

# Display the first 5 rows of our resulting DataFrame to preview the data.
first_result.head()


Percentage of answered questions: 100.0%
Number of questions: 134719


,q_id,time_to_answer
0,48382183,-132444692
1,48118024,0
2,48278223,0
3,48287348,0
4,48508974,0


In [5]:
# The INNER JOIN above silently dropped every question that had no answer,
# which is why we (wrongly) saw 100% answered and far too few questions.
# Switching to a LEFT JOIN keeps ALL questions from posts_questions (the left table),
# regardless of whether a matching answer exists. Unanswered questions get a NULL
# time_to_answer, so the answered-percentage falls below 100% and the row count climbs.
correct_query = """ 
                # SELECT the question ID
                SELECT q.id AS q_id,
                    # Time (in seconds) from the question to its VERY FIRST answer.
                    # NULL for questions that were never answered.
                    MIN(TIMESTAMP_DIFF(a.creation_date, q.creation_date, SECOND)) as time_to_answer
                # From the 'posts_questions' table (aliased as 'q')
                FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
                    # LEFT JOIN keeps every question even when it has no matching answer
                    LEFT JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
                # The connection point: the answer's parent_id must match the question's id
                ON q.id = a.parent_id
                # Only look at questions asked between January 1, 2018 and January 31, 2018
                WHERE q.creation_date >= '2018-01-01' and q.creation_date < '2018-02-01'
                # Group the results by question ID so we find the MIN time per unique question
                GROUP BY q_id
                # Sort the results by the fastest answer time
                ORDER BY time_to_answer
                """

# Run the query, and return a pandas DataFrame.
correct_result = client.query(correct_query).result().to_dataframe(create_bqstorage_client=False)
print("Percentage of answered questions: %s%%" % \
      (sum(correct_result["time_to_answer"].notnull()) / len(correct_result) * 100))
print("Number of questions:", len(correct_result))   


Percentage of answered questions: 83.3368387192557%
Number of questions: 161656


In [6]:
three_tables_query = """
                     -- 1. What columns do we want in our final table?
                     -- We want the user ID, and the EARLIEST (MIN) question and answer dates.
                     SELECT 
                         u.id AS id,
                         MIN(q.creation_date) AS q_creation_date,
                         MIN(a.creation_date) AS a_creation_date
                         
                     -- 2. What is our "Base" table? 
                     -- We start with the users table because it has our master list of accounts.
                     FROM `bigquery-public-data.stackoverflow.users` AS u
                     
                     -- 3. Attach the Questions table
                     -- We use a LEFT JOIN so we don't lose users who never asked a question.
                     LEFT JOIN `bigquery-public-data.stackoverflow.posts_questions` AS q
                         ON u.id = q.owner_user_id
                         
                     -- 4. Attach the Answers table
                     -- We use another LEFT JOIN so we don't lose users who never gave an answer.
                     LEFT JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
                         ON u.id = a.owner_user_id
                         
                     -- 5. Filter our Base list
                     -- We ONLY want to look at users who created their account in January 2019.
                     WHERE u.creation_date >= '2019-01-01' AND u.creation_date < '2019-02-01'
                     
                     -- 6. Group the results
                     -- Because we are calculating a MIN() date per user, we have to tell 
                     -- SQL to group all the raw data by the user's ID before finding the minimum.
                     GROUP BY id
                     """

# Run the query, and convert the results to a pandas DataFrame.
# Note: Remember to keep create_bqstorage_client=False to bypass the Storage API permission error
three_tables_result = client.query(three_tables_query).result().to_dataframe(create_bqstorage_client=False)

# Print the total number of users retrieved.
print("Number of users:", len(three_tables_result))

# Display the first 5 rows.
three_tables_result.head()


Number of users: 141706


,id,q_creation_date,a_creation_date
0,10990838,2019-02-24 21:10:52.273000+00:00,2020-07-15 12:03:29.830000+00:00
1,10922495,2019-01-16 13:05:07.587000+00:00,NaT
2,10976173,NaT,NaT
3,10989955,NaT,NaT
4,10993691,NaT,NaT


In [ ]:
# Query to find all distinct users who posted on January 1, 2019.
all_users_query = """
                  -- Get user IDs of everyone who ASKED a question on Jan 1, 2019
                  SELECT owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_questions`
                  WHERE EXTRACT(DATE FROM creation_date) = '2019-01-01'

                  -- UNION DISTINCT combines both result sets and automatically
                  -- removes any duplicate user IDs
                  UNION DISTINCT

                  -- Get user IDs of everyone who ANSWERED a question on Jan 1, 2019
                  SELECT owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_answers`
                  WHERE EXTRACT(DATE FROM creation_date) = '2019-01-01'
                  """

: 